# Build the corpus tables

Organises everything we know about the corpus as [tidy data](https://social-media-lab.net/processing/): each variable is a column, each observation is a row, and each type of observational unit is its own table.

| Table | One row per | Key | Links to posts through |
|---|---|---|---|
| `posts.csv` (metadata table) | TikTok post | `Post ID` | |
| `texts.csv` (text table) | Text of a post: caption, transcription or OCR | `Text ID` | `Post ID` |
| `images.csv` (images table) | Cover or extracted frame | `Image ID` | `Post ID` |

The tables are written to `/corpus/derived/tables/`. Run this notebook at any point, and again after each preprocessing step: it rebuilds all tables from what exists so far, so nothing is lost by re-running.

**Don't edit these tables by hand.** Save your own variables, such as codings or model output, in a separate table with `Text ID`, `Image ID` or `Post ID`, and merge when you need them.

In [ ]:
import json, os, re
from pathlib import Path

import pandas as pd

RAW = Path(os.environ.get("SEMINAR_RAW", "/corpus/raw"))
OUTPUT = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived"))
TRANSCRIPTS = OUTPUT / "transcripts"
FRAMES = OUTPUT / "frames"
OCR = OUTPUT / "ocr"
TABLES = OUTPUT / "tables"

VIDEO_SUFFIXES = {".mp4", ".mov", ".webm", ".mkv", ".m4v"}
COVER_SUFFIX = "_cover.jpg"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}


def visible(root, pattern):
    """Files under root, ignoring hidden folders such as Jupyter's .ipynb_checkpoints."""
    return sorted(p for p in root.rglob(pattern) if not any(part.startswith(".") for part in p.relative_to(root).parts))

## Metadata table

One row per post, from the JSON file collected with each video. Data contributors are deliberately not part of the corpus.

In [ ]:
def number(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def read_post(path):
    """Return the post stored in a JSON file, or None if it isn't a TikTok post."""
    try:
        post = json.loads(path.read_text(encoding="utf-8"))
    except (ValueError, OSError):
        return None
    return post if isinstance(post, dict) and post.get("id") and isinstance(post.get("author"), dict) else None


found = {}
for path in visible(RAW, "*"):
    if not path.is_file() or path.name.startswith((".", "_")):
        continue
    if path.name.endswith(COVER_SUFFIX):
        found.setdefault(path.with_name(path.name[: -len(COVER_SUFFIX)]), {})["cover"] = path
    elif path.suffix.lower() in IMAGE_SUFFIXES and re.fullmatch(r".+_\d+", path.stem):
        found.setdefault(path.with_name(path.stem.rsplit("_", 1)[0]), {}).setdefault("gallery", []).append(path)
    elif path.suffix.lower() in VIDEO_SUFFIXES:
        found.setdefault(path.with_suffix(""), {})["video"] = path
    elif path.suffix.lower() == ".json" and read_post(path):
        found.setdefault(path.with_suffix(""), {})["json"] = path

rows = []
captions = {}
for stem, files in found.items():
    post = read_post(files["json"]) if "json" in files else {}
    source = str(stem.relative_to(RAW))
    author, stats, music = post.get("author") or {}, post.get("stats") or {}, post.get("music") or {}
    created = number(post.get("createTime"))
    rows.append({
        "Post ID": str(post.get("id") or stem.name),
        "Username": author.get("uniqueId") or (stem.parent.name if "json" not in files else None),
        "Created": pd.to_datetime(created, unit="s", utc=True).strftime("%Y-%m-%d %H:%M:%S") if created else None,
        "Duration": number((post.get("video") or {}).get("duration")),
        "Plays": number(stats.get("playCount")),
        "Likes": number(stats.get("diggCount")),
        "Comments": number(stats.get("commentCount")),
        "Shares": number(stats.get("shareCount")),
        "Saves": number(stats.get("collectCount")),
        "Music Title": music.get("title"),
        "Music Original": music.get("original"),
        "Is Ad": post.get("isAd"),
        "Source": source,
        "Video Path": files.get("video") and str(files["video"]),
        "Cover Path": files.get("cover") and str(files["cover"]),
        "Metadata Path": files.get("json") and str(files["json"]),
    })
    captions[source] = (post.get("desc") or "").strip()

posts = pd.DataFrame(rows).sort_values("Post ID")
for column in ["Duration", "Plays", "Likes", "Comments", "Shares", "Saves"]:
    posts[column] = posts[column].astype("Int64")

post_by_source = posts.set_index("Source")[["Post ID", "Username"]].to_dict("index")
print(f"{len(posts)} posts")
print(f"{posts['Video Path'].notna().sum()} with video, {posts['Cover Path'].notna().sum()} with cover")

## Text table

One row per text, following the `Text Type` values Caption, Transcription and OCR. OCR has one row per extraction strategy, in the `Strategy` column, so strategies can be compared. Empty texts are left out.

In [ ]:
def source_key(source):
    return str(Path(source).with_suffix(""))


texts = []


def add_text(source, kind, text, strategy=None, language=None, model=None):
    post = post_by_source.get(source)
    if post is None or not text or not text.strip():
        return
    suffix = f"{kind.lower()}_{strategy}" if strategy else kind.lower()
    texts.append({
        "Text ID": f"{post['Post ID']}_{suffix}", "Post ID": post["Post ID"], "Username": post["Username"],
        "Text Type": kind, "Strategy": strategy, "Language": language, "Model": model, "Text": text.strip(),
    })


for source, caption in captions.items():
    add_text(source, "Caption", caption)

for path in visible(TRANSCRIPTS, "*.json"):
    record = json.loads(path.read_text(encoding="utf-8"))
    add_text(source_key(record["source"]), "Transcription", record["text"], language=record["language"], model=record["model"])

for path in visible(OCR, "*.json"):
    record = json.loads(path.read_text(encoding="utf-8"))
    add_text(source_key(record["source"]), "OCR", record["text"], strategy=record["strategy"], model="easyocr")

texts = pd.DataFrame(texts, columns=["Text ID", "Post ID", "Username", "Text Type", "Strategy", "Language", "Model", "Text"])
texts.groupby(["Text Type", "Strategy"], dropna=False).size().rename("Texts").reset_index()

## Images table

One row per image: the cover of each post, the images of photo posts, and every frame from `02_extract_frames`. Use `Strategy` to select the frames of one extraction strategy.

In [ ]:
images = []


def add_image(source, image_id, kind, strategy, frame, seconds, path):
    post = post_by_source[source]
    images.append({
        "Image ID": image_id, "Post ID": post["Post ID"], "Username": post["Username"], "Image Type": kind,
        "Strategy": strategy, "Frame": frame, "Time": seconds, "Path": path,
    })


for source, post in post_by_source.items():
    cover = posts.loc[posts["Source"] == source, "Cover Path"].iloc[0]
    if pd.notna(cover):
        add_image(source, f"{post['Post ID']}_cover", "Cover", None, None, None, cover)
    for path in found[RAW / source].get("gallery", []):
        add_image(source, f"{post['Post ID']}_{path.stem.rsplit('_', 1)[1]}", "Gallery", None, None, None, str(path))

for path in visible(FRAMES, "frames.json"):
    record = json.loads(path.read_text(encoding="utf-8"))
    source = source_key(record["source"])
    # The cover strategy repeats the cover already listed above.
    if source not in post_by_source or record["settings"]["method"] == "cover":
        continue
    post_id = post_by_source[source]["Post ID"]
    for frame in record["frames"]:
        add_image(source, f"{post_id}_{record['strategy']}_{frame['frame']:06d}", "Frame", record["strategy"], frame["frame"], frame["time"], str(path.parent / frame["file"]))

images = pd.DataFrame(images, columns=["Image ID", "Post ID", "Username", "Image Type", "Strategy", "Frame", "Time", "Path"])
images["Frame"] = images["Frame"].astype("Int64")
images.groupby(["Image Type", "Strategy"], dropna=False).size().rename("Images").reset_index()

## Check and save

Stops if a key isn't unique or a row can't be linked to a post, so a broken table is never saved.

In [ ]:
for table, key in [(posts, "Post ID"), (texts, "Text ID"), (images, "Image ID")]:
    assert table[key].is_unique, f"{key} is not unique"
for table, name in [(texts, "texts"), (images, "images")]:
    assert table["Post ID"].isin(posts["Post ID"]).all(), f"{name} has rows without a post"

TABLES.mkdir(parents=True, exist_ok=True)
for table, name in [(posts, "posts"), (texts, "texts"), (images, "images")]:
    partial = TABLES / f"{name}.csv.partial"
    table.to_csv(partial, index=False)
    partial.rename(TABLES / f"{name}.csv")
    print(f"{name}.csv: {len(table)} rows, {len(table.columns)} columns")

## Working with the tables

Always read IDs as text. TikTok IDs are too long for spreadsheets and are damaged if treated as numbers.

In [ ]:
IDS = {"Post ID": str, "Text ID": str, "Image ID": str}
posts = pd.read_csv(TABLES / "posts.csv", dtype=IDS)
texts = pd.read_csv(TABLES / "texts.csv", dtype=IDS)
images = pd.read_csv(TABLES / "images.csv", dtype=IDS)

# Example: how many texts of each type does each post have?
counts = texts.groupby(["Post ID", "Text Type"]).size().unstack(fill_value=0).reset_index()
posts[["Post ID", "Username", "Created", "Plays"]].merge(counts, on="Post ID", how="left").head()